# 🚛 FleetLogix — Avance 1: Generación de Datos Sintéticos

---

## ¿Qué hace este notebook?

Genera **505,000+ registros** distribuidos en 6 tablas de PostgreSQL, simulando **2 años de operación** de una empresa de logística.

| Tabla | Registros | Descripción |
|-------|-----------|-------------|
| `vehicles` | 200 | Flota de vehículos |
| `drivers` | 400 | Conductores |
| `routes` | 50 | Rutas entre ciudades |
| `trips` | 100,000 | Viajes realizados |
| `deliveries` | 400,000 | Paquetes entregados |
| `maintenance` | 5,000 | Registros de mantenimiento |

## Adaptaciones

1. **Ciudades de México (CDMX, GDL, MTY, PUE, TIJ)
2. **Locale `es_MX`** en Faker para nombres y direcciones mexicanas
3. **Distribución horaria realista** de los viajes (más tráfico en horas pico)
4. **Tipos de vehículo** adaptados al mercado mexicano con pesos corregidos
5. **Exportación a CSV** además de inserción directa a PostgreSQL
6. **Barra de progreso** con `tqdm` para monitorear la generación
7. **Reporte final en JSON** con estadísticas de calidad de datos

## Celda 1 — Instalación de dependencias

In [1]:
# Instala las librerías necesarias
# El símbolo ! al inicio ejecuta comandos de terminal desde el notebook
!pip install psycopg2-binary faker pandas numpy tqdm

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## Celda 2 — Importaciones y Configuración

In [2]:
# ── Librerías estándar de Python ──────────────────────────────
import json                          # Para guardar el reporte final en formato JSON
import logging                       # Para imprimir mensajes con timestamp
import random                        # Números aleatorios básicos
from datetime import datetime, timedelta  # Manejo de fechas y horas

# ── Librerías de datos ────────────────────────────────────────
import numpy as np                   # Operaciones numéricas y distribuciones
import pandas as pd                  # DataFrames (tablas en memoria)
from faker import Faker              # Generador de datos falsos realistas
from tqdm import tqdm                # Barras de progreso (visual al generar miles de registros)

# ── Conexión a PostgreSQL ─────────────────────────────────────
import psycopg2                      # Driver de Python para PostgreSQL
from psycopg2.extras import execute_batch  # Inserción eficiente en lotes

# ── Sistema de archivos ───────────────────────────────────────
import os                            # Para crear carpetas (os.makedirs)

# =============================================================
# CONFIGURACIÓN —
# =============================================================

# Datos de conexión la base de datos PostgreSQL local
DB_CONFIG = {
    'host':     'localhost',            # PostgreSQL corre en esta máquina
    'database': 'postgres',             # Nombre de la base de datos
    'user':     'postgres',             # Usuario de PostgreSQL
    'password': 'TU_PASSWORD',               # ← contraseña real
    'port':     5432                    # Puerto por defecto de PostgreSQL
}

LOAD_TO_DB = True    # True → inserta en PostgreSQL  |  False → solo genera CSVs
SAVE_CSV   = True    # True → guarda archivos CSV en la carpeta ./data/
CSV_FOLDER = './data'

# =============================================================
# SEMILLA DE REPRODUCIBILIDAD
# Con el mismo SEED, siempre se generan los mismos datos.
# Útil para que todos los del equipo tengan
# exactamente la misma base de datos.
# =============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Faker con locale mexicano: genera nombres, direcciones y teléfonos
# en formato mexicano.
fake = Faker('es_MX')
Faker.seed(SEED)

# =============================================================
# CONFIGURACIÓN DEL LOGGER
# Escribe mensajes tanto en consola como en un archivo .log
# para poder revisar qué pasó si algo falla.
# =============================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('data_generation.log'),  # Guarda en archivo
        logging.StreamHandler()                       # Imprime en consola
    ]
)

print('✅ Librerías importadas y configuración lista')
print(f'   LOAD_TO_DB = {LOAD_TO_DB}')
print(f'   SAVE_CSV   = {SAVE_CSV}')
print(f'   CSV_FOLDER = {CSV_FOLDER}')

✅ Librerías importadas y configuración lista
   LOAD_TO_DB = True
   SAVE_CSV   = True
   CSV_FOLDER = ./data


## Celda 3 — Clase `DataGenerator`

Esta celda define toda la lógica de generación de datos. **No produce output visible** — solo carga la clase en memoria para usarla en celdas posteriores.

In [3]:
class DataGenerator:
    """
    Clase principal que encapsula toda la lógica de generación de datos.

    ¿Por qué usar una CLASE en lugar de funciones sueltas?
    - Agrupa la conexión a la BD y todos los datos generados en un solo objeto
    - self.connection y self.cursor se comparten entre todos los métodos
      sin necesidad de pasarlos como parámetros cada vez
    - self.dataframes guarda los DataFrames para exportar a CSV después
    - Facilita el manejo de errores con try/finally en main()

    El script de la escuela usa exactamente esta estructura;
    nosotros la completamos con las partes que dicen #TO DO#
    """

    def __init__(self, db_config):
        """
        Constructor: se ejecuta cuando escribimos DataGenerator(DB_CONFIG).
        Inicializa los atributos del objeto pero NO abre la conexión todavía.
        """
        self.db_config  = db_config
        self.connection = None       # Se asigna en connect()
        self.cursor     = None       # Se asigna en connect()
        self.dataframes = {}         # Diccionario para guardar los DataFrames generados

        # Ciudades de operación — ADAPTACIÓN: cambiamos Colombia → México
        # El script original usa: ['Bogotá','Medellín','Cali','Barranquilla','Cartagena']
        self.cities = [
            'Ciudad de México', 'Guadalajara', 'Monterrey', 'Puebla', 'Tijuana'
        ]

        # Contadores para el reporte final
        self.counters = {
            'vehicles': 0, 'drivers': 0, 'routes': 0,
            'trips': 0, 'deliveries': 0, 'maintenance': 0
        }

    # ─────────────────────────────────────────────────────────
    # CONEXIÓN A POSTGRESQL
    # ─────────────────────────────────────────────────────────

    def connect(self):
        """Abre la conexión a PostgreSQL. Retorna True si tuvo éxito."""
        try:
            self.connection = psycopg2.connect(**self.db_config)
            # **self.db_config desempaqueta el diccionario como argumentos:
            # equivale a: psycopg2.connect(host='localhost', database='fleetlogix', ...)
            self.cursor = self.connection.cursor()
            logging.info('✅ Conexión exitosa a PostgreSQL')
            return True
        except Exception as e:
            logging.error(f'❌ Error al conectar: {e}')
            return False

    def close(self):
        """Cierra la conexión. Se llama en el bloque finally para garantizar cierre."""
        if self.cursor:
            self.cursor.close()
        if self.connection:
            self.connection.close()
        logging.info('🔒 Conexión cerrada')

    # ─────────────────────────────────────────────────────────
    # TABLA 1: VEHICLES
    # ─────────────────────────────────────────────────────────

    def generate_vehicles(self, count=200):
        """
        Genera 200 vehículos con tipos y capacidades realistas.

        ADAPTACIONES vs script de la escuela:
        - Placa formato mexicano (ABC-123) en lugar de colombiana (ABC123)
        - Estado 'maintenance' en lugar de 'inactive' para el 10% inactivos
        - La query INSERT estaba como #TO DO# → la completamos

        TIPOS DE VEHÍCULO con pesos (probabilidad de selección):
        ('Nombre', capacidad_kg, combustible, probabilidad)
        Las probabilidades deben sumar 1.0
        """
        logging.info(f'Generando {count} vehículos...')

        # Catálogo de tipos de vehículo
        # Formato: (tipo, capacidad_kg, combustible, probabilidad_de_selección)
        vehicle_types = [
            ('Camión Grande',  5000, 'Diésel',   0.30),
            ('Camión Mediano', 3000, 'Diésel',   0.30),
            ('Van',            1500, 'Gasolina', 0.30),
            ('Motocicleta',      50, 'Gasolina', 0.10),
        ]

        vehicles    = []      # Lista de tuplas para insertar en la BD
        placas_vistas = set() # Set para garantizar placas únicas

        for i in range(count):
            # random.choices con weights elige según probabilidades
            # [0] porque choices devuelve lista, queremos el primer elemento
            v_type, capacity, fuel, _ = random.choices(
                vehicle_types,
                weights=[vt[3] for vt in vehicle_types]  # Extrae las probabilidades
            )[0]

            # Placa mexicana formato: ABC-123
            # fake.random_uppercase_letter() → letra aleatoria A-Z
            while True:
                placa = (
                    f"{fake.random_uppercase_letter()}"
                    f"{fake.random_uppercase_letter()}"
                    f"{fake.random_uppercase_letter()}"
                    f"-{random.randint(100, 999)}"
                ).replace('Ñ', 'N')  # Las placas no usan Ñ
                if placa not in placas_vistas:
                    placas_vistas.add(placa)
                    break

            # Fecha de adquisición entre hace 5 años y hace 1 mes
            acquisition_date = fake.date_between(start_date='-5y', end_date='-1m')

            # 90% activos, 10% en mantenimiento
            # random.choices(['active']*9 + ['maintenance']) = 90%/10%
            status = random.choice(['active'] * 9 + ['maintenance'])

            # La capacidad tiene variación ±20% para hacerla más realista
            real_capacity = round(capacity * random.uniform(0.8, 1.2), 2)

            # Tupla en el MISMO orden que las columnas del INSERT
            vehicles.append((
                placa,
                v_type,
                real_capacity,
                fuel,
                acquisition_date,
                status
            ))

        # ── Guardar como DataFrame para CSV ──────────────────
        cols = ['license_plate','vehicle_type','capacity_kg',
                'fuel_type','acquisition_date','status']
        df = pd.DataFrame(vehicles, columns=cols)
        df.index = df.index + 1           # vehicle_id empieza en 1
        df.index.name = 'vehicle_id'
        self.dataframes['vehicles'] = df

        # ── Insertar en PostgreSQL (si está activo) ───────────
        if self.cursor:
            # QUERY COMPLETADA — el script de la escuela tiene aquí #TO DO#
            # Los %s son placeholders: psycopg2 los sustituye de forma segura
            # evitando SQL injection (nunca concatenar strings en queries SQL)
            query = """
                INSERT INTO vehicles
                    (license_plate, vehicle_type, capacity_kg,
                     fuel_type, acquisition_date, status)
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            # execute_batch inserta en lotes de page_size filas
            # Es ~10x más rápido que hacer INSERT uno a uno
            execute_batch(self.cursor, query, vehicles, page_size=100)
            self.connection.commit()  # Confirmar la transacción

        self.counters['vehicles'] = count
        logging.info(f'✅ {count} vehículos generados')

    # ─────────────────────────────────────────────────────────
    # TABLA 2: DRIVERS
    # ─────────────────────────────────────────────────────────

    def generate_drivers(self, count=400):
        """
        Genera 400 conductores con datos mexicanos realistas.

        ADAPTACIONES vs script de la escuela:
        - Teléfonos con prefijo mexicano (55-XXXXXXXX) en lugar de colombiano (3XXXXXXXXX)
        - Licencia tipo 'C' (México) en lugar de 'C1/C2/C3/A2' (Colombia)
        - 95% activos (igual que el script original)
        """
        logging.info(f'Generando {count} conductores...')

        # Tipos de licencia de conducir en México para transporte de carga
        license_types = ['C', 'D', 'E']  # C=carga federal, D=articulado, E=pasajeros

        drivers     = []
        emp_usados  = set()  # Para códigos de empleado únicos
        lic_usados  = set()  # Para números de licencia únicos

        for i in range(count):
            # Código de empleado secuencial con ceros a la izquierda
            # zfill(4) rellena con ceros: 1 → '0001', 12 → '0012'
            employee_code = f'EMP{str(i+1).zfill(4)}'  # EMP0001, EMP0002, ...

            first_name = fake.first_name()   # Nombre mexicano
            last_name  = fake.last_name() + ' ' + fake.last_name()  # 2 apellidos

            # Número de licencia mexicana (10 dígitos numéricos)
            while True:
                lic_num = str(random.randint(1_000_000_000, 9_999_999_999))
                if lic_num not in lic_usados:
                    lic_usados.add(lic_num)
                    break

            license_type   = random.choice(license_types)
            # Licencia válida entre el próximo mes y 3 años en el futuro
            license_expiry = fake.date_between(start_date='+1m', end_date='+3y')

            # Teléfono mexicano: 55 + 8 dígitos = 10 dígitos totales
            # fake.numerify('########') genera 8 dígitos aleatorios
            phone = f'55{fake.numerify("########")}'

            # Contratado hace entre 5 años y hace 1 semana
            hire_date = fake.date_between(start_date='-5y', end_date='-1w')

            # 95% activos (19 de cada 20)
            status = random.choice(['active'] * 19 + ['inactive'])

            drivers.append((
                employee_code, first_name, last_name,
                lic_num, license_expiry, phone, hire_date, status
            ))

        # Guardar como DataFrame
        cols = ['employee_code','first_name','last_name',
                'license_number','license_expiry','phone','hire_date','status']
        df = pd.DataFrame(drivers, columns=cols)
        df.index = df.index + 1
        df.index.name = 'driver_id'
        self.dataframes['drivers'] = df

        if self.cursor:
            # Query del script original (ya estaba completa)
            query = """
                INSERT INTO drivers
                    (employee_code, first_name, last_name,
                     license_number, license_expiry, phone, hire_date, status)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            execute_batch(self.cursor, query, drivers, page_size=100)
            self.connection.commit()

        self.counters['drivers'] = count
        logging.info(f'✅ {count} conductores generados')

    # ─────────────────────────────────────────────────────────
    # TABLA 3: ROUTES
    # ─────────────────────────────────────────────────────────

    def _get_distance(self, origin, destination):
        """
        Distancias reales aproximadas en km entre las 5 ciudades mexicanas.

        ADAPTACIÓN: el script original tiene distancias colombianas.
        tuple(sorted([a,b])) ordena alfabéticamente para que
        ('Monterrey','CDMX') y ('CDMX','Monterrey') usen la misma clave.
        """
        distances = {
            ('Ciudad de México', 'Guadalajara'):   545,
            ('Ciudad de México', 'Monterrey'):     940,
            ('Ciudad de México', 'Puebla'):        130,
            ('Ciudad de México', 'Tijuana'):      2780,
            ('Guadalajara',      'Monterrey'):     800,
            ('Guadalajara',      'Puebla'):        500,
            ('Guadalajara',      'Tijuana'):      2250,
            ('Monterrey',        'Puebla'):       1000,
            ('Monterrey',        'Tijuana'):      1670,
            ('Puebla',           'Tijuana'):      2850,
        }
        # sorted ordena las ciudades para que A→B y B→A usen la misma clave
        key = tuple(sorted([origin, destination]))
        return distances.get(key, 500)  # 500 km como valor por defecto

    def generate_routes(self, count=50):
        """
        Genera 50 rutas entre las 5 ciudades mexicanas.
        El script original de la escuela usa lógica idéntica;
        solo cambiamos ciudades y peajes (pesos mexicanos en lugar de COP).
        """
        logging.info(f'Generando {count} rutas...')

        routes        = []
        route_counter = 1

        # Genera combinaciones A→B entre todas las ciudades
        for origin in self.cities:
            for destination in self.cities:
                if origin == destination:
                    continue  # No hay ruta de una ciudad a sí misma

                # Más rutas desde/hacia CDMX (es el hub principal)
                num_routes = 3 if ('Ciudad de México' in [origin, destination]) else 2

                for _ in range(num_routes):
                    route_code   = f'R{str(route_counter).zfill(3)}'  # R001, R002...
                    base_distance = self._get_distance(origin, destination)

                    # Variación de ±50 km (rutas alternativas entre las mismas ciudades)
                    distance = base_distance + random.uniform(-50, 50)

                    # Tiempo estimado = distancia / velocidad promedio (60-80 km/h)
                    avg_speed = random.uniform(60, 80)
                    duration  = distance / avg_speed  # en horas

                    # Peaje en pesos mexicanos: ~$20 por cada 100 km
                    # El script original usa pesos colombianos ($15,000 COP/100km)
                    toll_cost = int(distance / 100) * 200  # $200 MXN por cada 100 km

                    routes.append((
                        route_code,
                        origin,
                        destination,
                        round(distance, 2),
                        round(duration, 2),
                        toll_cost
                    ))

                    route_counter += 1
                    if route_counter > count:
                        break
                if route_counter > count:
                    break
            if route_counter > count:
                break

        routes = routes[:count]  # Exactamente count rutas

        # Guardar como DataFrame
        cols = ['route_code','origin_city','destination_city',
                'distance_km','estimated_duration_hours','toll_cost']
        df = pd.DataFrame(routes, columns=cols)
        df.index = df.index + 1
        df.index.name = 'route_id'
        self.dataframes['routes'] = df

        if self.cursor:
            # Query ya estaba completa en el script original
            query = """
                INSERT INTO routes
                    (route_code, origin_city, destination_city,
                     distance_km, estimated_duration_hours, toll_cost)
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            execute_batch(self.cursor, query, routes, page_size=50)
            self.connection.commit()

        self.counters['routes'] = len(routes)
        logging.info(f'✅ {len(routes)} rutas generadas')

    # ─────────────────────────────────────────────────────────
    # TABLA 4: TRIPS
    # ─────────────────────────────────────────────────────────

    def _get_hourly_distribution(self):
        """
        Distribución de probabilidad de salidas por hora del día.
        Más viajes en horario laboral (6am-8pm), picos a las 8-12 y 14-18.
        El resultado debe sumar exactamente 1.0 para usarse como probabilidad.

        Esta función es idéntica a la del script de la escuela.
        """
        probs = np.ones(24) * 0.02   # Base: 2% por hora (distribución uniforme)
        probs[6:20]  = 0.06          # 6am-8pm: más actividad (6%)
        probs[8:12]  = 0.08          # 8am-12pm: pico mañana (8%)
        probs[14:18] = 0.07          # 2pm-6pm: pico tarde (7%)
        return probs / probs.sum()   # Normalizar para que sumen 1.0

    def generate_trips(self, count=100_000):
        """
        Genera 100,000 viajes distribuidos en 2 años de operación.

        DIFERENCIA CLAVE vs script de la escuela:
        El original consulta la BD para obtener vehículos/conductores/rutas.
        Nosotros los obtenemos de self.dataframes (los DataFrames en memoria),
        lo que permite funcionar SIN conexión a PostgreSQL (modo CSV).

        LÓGICA DE DISTRIBUCIÓN TEMPORAL:
        Los 100,000 viajes se distribuyen en 730 días (2 años).
        Cada viaje avanza la fecha en: 730 días * 1440 min/día / 100,000 = ~10.5 min
        Esto garantiza que los viajes estén distribuidos uniformemente en el tiempo.
        """
        logging.info(f'Generando {count:,} viajes...')

        # ── Obtener IDs válidos ───────────────────────────────
        # Si hay conexión a BD, consultamos directamente
        if self.cursor:
            self.cursor.execute(
                "SELECT vehicle_id, capacity_kg FROM vehicles WHERE status = 'active'"
            )
            vehicles_data = self.cursor.fetchall()  # Lista de (id, capacidad)

            self.cursor.execute("SELECT driver_id FROM drivers WHERE status = 'active'")
            driver_ids = [d[0] for d in self.cursor.fetchall()]

            self.cursor.execute(
                "SELECT route_id, distance_km, estimated_duration_hours FROM routes"
            )
            routes_data = self.cursor.fetchall()  # Lista de (id, distancia, duración)

        else:
            # Sin conexión: usamos los DataFrames generados en memoria
            vdf = self.dataframes['vehicles']
            vehicles_data = [
                (idx, row['capacity_kg'])
                for idx, row in vdf[vdf['status'] == 'active'].iterrows()
            ]
            ddf = self.dataframes['drivers']
            driver_ids = ddf[ddf['status'] == 'active'].index.tolist()

            rdf = self.dataframes['routes']
            routes_data = [
                (idx, row['distance_km'], row['estimated_duration_hours'])
                for idx, row in rdf.iterrows()
            ]

        # Fecha de inicio: 2 años atrás desde hoy
        start_date   = datetime.now() - timedelta(days=730)
        current_date = start_date

        trips = []

        # tqdm muestra una barra de progreso: [=====>    ] 50%
        for i in tqdm(range(count), desc='Generando trips'):
            vehicle_id, capacity = random.choice(vehicles_data)
            capacity             = float(capacity)
            driver_id            = random.choice(driver_ids)
            route_id, distance, est_duration = random.choice(routes_data)
            distance             = float(distance)
            est_duration         = float(est_duration)

            # Hora de salida según distribución realista (más salidas de 8am-6pm)
            hour = int(np.random.choice(range(24), p=self._get_hourly_distribution()))
            departure = current_date.replace(
                hour=hour,
                minute=random.randint(0, 59),
                second=0,
                microsecond=0
            )

            # Duración real = estimada ±30% (tráfico, descansos, condiciones)
            actual_duration = est_duration * random.uniform(0.8, 1.3)
            arrival         = departure + timedelta(hours=actual_duration)

            # Consumo de combustible: 8-15 litros por 100 km
            fuel_consumed = distance * random.uniform(0.08, 0.15)

            # Peso total: entre 40% y 90% de la capacidad del vehículo
            total_weight = capacity * random.uniform(0.4, 0.9)

            # Estado: si el viaje ya pasó → completado; si no → en progreso
            if arrival < datetime.now():
                status  = 'completed'
                arr_val = arrival
            else:
                status  = 'in_progress'
                arr_val = None  # NULL en la BD: todavía no ha llegado

            trips.append((
                vehicle_id,
                driver_id,
                route_id,
                departure,
                arr_val,
                round(fuel_consumed, 2),
                round(total_weight, 2),
                status
            ))

            # Avanzar fecha uniformemente: distribuye los viajes en 2 años
            # 730 días * 1440 minutos/día / count viajes = minutos por viaje
            current_date += timedelta(minutes=int(1440 * 730 / count))

        # Guardar DataFrame
        cols = ['vehicle_id','driver_id','route_id','departure_datetime',
                'arrival_datetime','fuel_consumed_liters','total_weight_kg','status']
        df = pd.DataFrame(trips, columns=cols)
        df.index = df.index + 1
        df.index.name = 'trip_id'
        self.dataframes['trips'] = df

        if self.cursor:
            query = """
                INSERT INTO trips
                    (vehicle_id, driver_id, route_id, departure_datetime,
                     arrival_datetime, fuel_consumed_liters, total_weight_kg, status)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            # Inserción en lotes de 1,000 filas para no sobrecargar la memoria
            batch_size = 1000
            for i in range(0, len(trips), batch_size):
                execute_batch(self.cursor, query, trips[i:i+batch_size], page_size=100)
                self.connection.commit()
                if i % 10_000 == 0:
                    logging.info(f'  Progreso trips: {i:,}/{count:,}')

        self.counters['trips'] = count
        logging.info(f'✅ {count:,} viajes generados')

    # ─────────────────────────────────────────────────────────
    # TABLA 5: DELIVERIES
    # ─────────────────────────────────────────────────────────

    def _distribute_weight(self, total_weight, num_packages):
        """
        Distribuye el peso total del viaje entre los paquetes.
        Usa distribución exponencial para que haya paquetes de distintos tamaños
        (más realista que dividir el peso en partes iguales).

        Esta función es idéntica a la del script de la escuela.
        """
        # np.random.exponential genera valores con distribución asimétrica
        # (muchos paquetes pequeños, pocos paquetes grandes)
        weights = np.random.exponential(scale=1.0, size=num_packages)
        # Normalizar: dividir cada peso por la suma → suman 1.0
        weights = weights / weights.sum() * total_weight * 0.95  # 95% del peso total
        # Garantizar mínimo 0.5 kg por paquete
        weights = np.maximum(weights, 0.5)
        return weights

    def generate_deliveries(self, count=400_000):
        """
        Genera ~400,000 entregas distribuidas entre los viajes existentes.

        COMPLETAMOS EL #TO DO# del script de la escuela:
        La query que obtiene los datos de trips estaba pendiente.
        Necesitamos: trip_id, departure, arrival, total_weight y ciudad de destino
        (para asignar la dirección del cliente a esa ciudad).

        LÓGICA:
        - Cada viaje genera entre 2 y 6 entregas (promedio 4)
        - El tracking number es único: FL2024XXXXXXXX
        - 90% se entregan a tiempo, 10% con retraso
        - 95% de las entregas completadas tienen firma
        """
        logging.info(f'Generando ~{count:,} entregas...')

        # ── QUERY COMPLETADA (era #TO DO# en el script de la escuela) ──────
        # Necesitamos: trip_id, departure, arrival, peso total y ciudad destino
        # La ciudad destino viene de routes.destination_city (JOIN necesario)
        if self.cursor:
            self.cursor.execute("""
                SELECT
                    t.trip_id,
                    t.departure_datetime,
                    t.arrival_datetime,
                    t.total_weight_kg,
                    r.destination_city
                FROM trips t
                JOIN routes r ON t.route_id = r.route_id
                ORDER BY t.trip_id
            """)
            trips_data = self.cursor.fetchall()  # Lista de tuplas
        else:
            # Sin conexión: construir desde DataFrames en memoria
            tdf = self.dataframes['trips'].reset_index()  # trip_id como columna
            rdf = self.dataframes['routes'].reset_index()  # route_id como columna
            merged = tdf.merge(rdf[['route_id','destination_city']], on='route_id')
            trips_data = [
                (row['trip_id'], row['departure_datetime'],
                 row['arrival_datetime'], row['total_weight_kg'],
                 row['destination_city'])
                for _, row in merged.iterrows()
            ]

        deliveries       = []
        delivery_counter = 0
        tracking_usados  = set()  # Para tracking numbers únicos

        for trip_id, departure, arrival, total_weight, city in tqdm(
            trips_data, desc='Generando deliveries'
        ):
            # Entre 2 y 6 entregas por viaje (promedio 4 → ~400k total)
            # Las probabilidades: 10% de 2 entregas, 20% de 3, 40% de 4, etc.
            num_deliveries = int(np.random.choice(
                [2, 3, 4, 5, 6],
                p=[0.10, 0.20, 0.40, 0.20, 0.10]
            ))

            # Distribuir el peso total entre los paquetes de este viaje
            if total_weight and not pd.isna(total_weight):
                weights = self._distribute_weight(float(total_weight), num_deliveries)
            else:
                weights = [random.uniform(0.5, 10)] * num_deliveries

            # Tiempo disponible para las entregas del viaje
            if arrival and not pd.isna(arrival):
                if isinstance(arrival, str):
                    arrival = datetime.fromisoformat(arrival)
                if isinstance(departure, str):
                    departure = datetime.fromisoformat(departure)
                delivery_duration  = (arrival - departure).total_seconds() / 3600
                time_per_delivery  = delivery_duration / num_deliveries
            else:
                time_per_delivery = 0.5  # 30 min por entrega si no hay arrival

            for i in range(num_deliveries):
                # Tracking number único: FL + año + contador de 8 dígitos
                while True:
                    tn = f'FL{datetime.now().year}{str(delivery_counter+1).zfill(8)}'
                    if tn not in tracking_usados:
                        tracking_usados.add(tn)
                        break
                    delivery_counter += 1  # Incrementar si hay colisión

                customer_name     = fake.name()
                # Dirección en la ciudad de destino del viaje
                delivery_address  = f'{fake.street_address()}, {city}'
                package_weight    = weights[i]

                # Hora programada: distribuida uniformemente en el trayecto
                if isinstance(departure, str):
                    departure = datetime.fromisoformat(departure)
                scheduled = departure + timedelta(hours=time_per_delivery * (i + 0.5))

                if arrival and not pd.isna(arrival):
                    # 90% entregados a tiempo, 10% con retraso de 1-3 horas
                    if random.random() < 0.90:
                        delivered = scheduled + timedelta(minutes=random.randint(-30, 30))
                    else:
                        delivered = scheduled + timedelta(minutes=random.randint(60, 180))
                    delivery_status = 'delivered'
                    signature       = random.random() < 0.95  # 95% con firma
                else:
                    delivered       = None
                    delivery_status = 'pending'
                    signature       = False

                deliveries.append((
                    trip_id, tn, customer_name, delivery_address,
                    round(float(package_weight), 2),
                    scheduled, delivered, delivery_status, signature
                ))
                delivery_counter += 1

                if delivery_counter >= count:
                    break
            if delivery_counter >= count:
                break

        # Guardar DataFrame
        cols = ['trip_id','tracking_number','customer_name','delivery_address',
                'package_weight_kg','scheduled_datetime','delivered_datetime',
                'delivery_status','recipient_signature']
        df = pd.DataFrame(deliveries, columns=cols)
        df.index = df.index + 1
        df.index.name = 'delivery_id'
        self.dataframes['deliveries'] = df

        if self.cursor:
            query = """
                INSERT INTO deliveries
                    (trip_id, tracking_number, customer_name, delivery_address,
                     package_weight_kg, scheduled_datetime, delivered_datetime,
                     delivery_status, recipient_signature)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            batch_size = 1000
            for i in range(0, len(deliveries), batch_size):
                execute_batch(self.cursor, query, deliveries[i:i+batch_size], page_size=100)
                self.connection.commit()
                if i % 50_000 == 0:
                    logging.info(f'  Progreso deliveries: {i:,}/{len(deliveries):,}')

        self.counters['deliveries'] = len(deliveries)
        logging.info(f'✅ {len(deliveries):,} entregas generadas')

    # ─────────────────────────────────────────────────────────
    # TABLA 6: MAINTENANCE
    # ─────────────────────────────────────────────────────────

    def generate_maintenance(self, count=5000):
        """
        Genera 5,000 registros de mantenimiento basados en el historial de viajes.

        COMPLETAMOS EL #TO DO# del script de la escuela:
        La query que obtiene estadísticas de vehículos estaba pendiente.
        Necesitamos: vehicle_id, tipo, total de viajes, primer y último viaje.
        Esto permite calcular cuántos mantenimientos corresponden a cada vehículo
        y distribuirlos en su período de operación real.

        ADAPTACIÓN en costos:
        El script original usa pesos colombianos (COP: 150,000 - 500,000).
        Nosotros usamos pesos mexicanos (MXN: multiplicamos ×0.04 aprox).
        """
        logging.info(f'Generando {count:,} registros de mantenimiento...')

        # ── QUERY COMPLETADA (era #TO DO# en el script de la escuela) ──────
        # Estadísticas por vehículo: cuántos viajes hizo y en qué período
        # MIN/MAX departure_datetime da el primer y último viaje del vehículo
        if self.cursor:
            self.cursor.execute("""
                SELECT
                    v.vehicle_id,
                    v.vehicle_type,
                    COUNT(t.trip_id)              AS trip_count,
                    MIN(t.departure_datetime)     AS first_trip,
                    MAX(t.departure_datetime)     AS last_trip
                FROM vehicles v
                LEFT JOIN trips t ON v.vehicle_id = t.vehicle_id
                GROUP BY v.vehicle_id, v.vehicle_type
                ORDER BY v.vehicle_id
            """)
            vehicle_stats = self.cursor.fetchall()
        else:
            # Sin conexión: calcular desde DataFrames
            vdf = self.dataframes['vehicles'].reset_index()
            tdf = self.dataframes['trips'].reset_index()
            stats = tdf.groupby('vehicle_id')['departure_datetime'].agg(
                trip_count='count', first_trip='min', last_trip='max'
            ).reset_index()
            merged = vdf.merge(stats, on='vehicle_id', how='left')
            vehicle_stats = [
                (row['vehicle_id'], row['vehicle_type'],
                 row.get('trip_count', 0), row.get('first_trip'), row.get('last_trip'))
                for _, row in merged.iterrows()
            ]

        # Catálogo de mantenimientos con costos en PESOS MEXICANOS (MXN)
        # Formato: (tipo, costo_base_mxn, días_hasta_siguiente)
        maintenance_types = [
            ('Cambio de aceite',        6_000,  30),   # ~$150 USD
            ('Revisión de frenos',     10_000,  60),   # ~$250 USD
            ('Cambio de llantas',      18_000,  90),   # ~$450 USD
            ('Mantenimiento general',  14_000,  45),   # ~$350 USD
            ('Revisión de motor',      20_000,  60),   # ~$500 USD
            ('Alineación y balanceo',   7_200,  30),   # ~$180 USD
        ]

        maintenance_records = []

        for vehicle_id, vehicle_type, trip_count, first_trip, last_trip in vehicle_stats:
            # Número de mantenimientos = 1 por cada 20 viajes (mínimo 1)
            # int() requiere que el valor no sea NaN.
            # pd.isna() detecta NaN/None; si es NaN lo tratamos como 0 viajes.
            _tc = 0 if (trip_count is None or (isinstance(trip_count, float) and pd.isna(trip_count))) else int(trip_count)
            num_maintenance = max(1, _tc // 20)

            if first_trip and last_trip and not pd.isna(first_trip):
                # Convertir a datetime si viene como string
                if isinstance(first_trip, str):
                    first_trip = datetime.fromisoformat(first_trip)
                if isinstance(last_trip, str):
                    last_trip = datetime.fromisoformat(last_trip)

                operation_days = (last_trip - first_trip).days

                for i in range(min(num_maintenance, count - len(maintenance_records))):
                    # Distribuir mantenimientos uniformemente en el período de operación
                    # i+1 / num_maintenance+1 → fracciones 1/n, 2/n, ..., n/n
                    days_offset = int(operation_days * (i + 1) / (num_maintenance + 1))
                    maint_date  = (first_trip + timedelta(days=days_offset)).date()

                    maint_type, base_cost, days_next = random.choice(maintenance_types)

                    # Costo con variación ±20%
                    cost = base_cost * random.uniform(0.8, 1.2)

                    description   = f'{maint_type} programado para {maint_date.strftime("%Y-%m-%d")}'
                    next_maint    = maint_date + timedelta(days=days_next)
                    performed_by  = f'{fake.first_name()} {fake.last_name()}'

                    maintenance_records.append((
                        vehicle_id, maint_date, maint_type, description,
                        round(cost, 2), next_maint, performed_by
                    ))

                    if len(maintenance_records) >= count:
                        break

        # Guardar DataFrame
        cols = ['vehicle_id','maintenance_date','maintenance_type','description',
                'cost','next_maintenance_date','performed_by']
        df = pd.DataFrame(maintenance_records[:count], columns=cols)
        df.index = df.index + 1
        df.index.name = 'maintenance_id'
        self.dataframes['maintenance'] = df

        if self.cursor:
            query = """
                INSERT INTO maintenance
                    (vehicle_id, maintenance_date, maintenance_type,
                     description, cost, next_maintenance_date, performed_by)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            execute_batch(
                self.cursor, query, maintenance_records[:count], page_size=100
            )
            self.connection.commit()

        self.counters['maintenance'] = min(len(maintenance_records), count)
        logging.info(f'✅ {self.counters["maintenance"]:,} mantenimientos generados')

    # ─────────────────────────────────────────────────────────
    # VALIDACIÓN DE CALIDAD DE DATOS
    # ─────────────────────────────────────────────────────────
    
    def validate_data_quality(self):
        # -----------------------------------
        """
        Verifica que los datos generados son coherentes e íntegros.
        Funciona tanto sobre la BD (si hay conexión) como sobre los DataFrames.

        Validaciones del script de la escuela (completamos las que faltaban):
        1. Trips sin vehículo válido (FK violation)
        2. Deliveries sin trip válido (FK violation)
        3. Trips con arrival < departure (error temporal)
        4. Trips con peso > capacidad del vehículo
        5. Entregas sin tracking number
        """
        logging.info('\n🔍 VALIDANDO CALIDAD DE DATOS...')

        if self.cursor:
            # Validaciones sobre PostgreSQL (idénticas al script de la escuela)
            validations = {
                'FK: Trips sin vehículo válido': """
                    SELECT COUNT(*) FROM trips t
                    LEFT JOIN vehicles v ON t.vehicle_id = v.vehicle_id
                    WHERE v.vehicle_id IS NULL
                """,
                'FK: Deliveries sin trip válido': """
                    SELECT COUNT(*) FROM deliveries d
                    LEFT JOIN trips t ON d.trip_id = t.trip_id
                    WHERE t.trip_id IS NULL
                """,
                'Temporal: arrival < departure': """
                    SELECT COUNT(*) FROM trips
                    WHERE arrival_datetime IS NOT NULL
                    AND arrival_datetime < departure_datetime
                """,
                'Peso: trips > capacidad vehículo': """
                    SELECT COUNT(*) FROM trips t
                    JOIN vehicles v ON t.vehicle_id = v.vehicle_id
                    WHERE t.total_weight_kg > v.capacity_kg
                """,
                'Deliveries sin tracking number': """
                    SELECT COUNT(*) FROM deliveries
                    WHERE tracking_number IS NULL OR tracking_number = ''
                """
            }
            all_valid = True
            for desc, q in validations.items():
                self.cursor.execute(q)
                n = self.cursor.fetchone()[0]
                icon = '✅' if n == 0 else '⚠️'
                logging.info(f'  {icon} {desc}: {n}')
                if n > 0:
                    all_valid = False
        else:
            # Validaciones sobre DataFrames (sin conexión a BD)
            all_valid = True
            vids = set(self.dataframes['vehicles'].index)
            dids = set(self.dataframes['drivers'].index)
            rids = set(self.dataframes['routes'].index)
            tids = set(self.dataframes['trips'].index)
            tdf  = self.dataframes['trips']
            deldf = self.dataframes['deliveries']

            checks = [
                ('FK: vehicle_id en trips',   (~tdf['vehicle_id'].isin(vids)).sum()),
                ('FK: driver_id en trips',    (~tdf['driver_id'].isin(dids)).sum()),
                ('FK: route_id en trips',     (~tdf['route_id'].isin(rids)).sum()),
                ('FK: trip_id en deliveries', (~deldf['trip_id'].isin(tids)).sum()),
                ('Sin tracking number',
                 deldf['tracking_number'].isna().sum() + (deldf['tracking_number']=='').sum()),
            ]
            for desc, n in checks:
                icon = '✅' if n == 0 else '⚠️'
                logging.info(f'  {icon} {desc}: {n}')
                if n > 0:
                    all_valid = False

        return all_valid

    # ─────────────────────────────────────────────────────────
    # EXPORTAR A CSV
    # ─────────────────────────────────────────────────────────

    def save_csv(self, folder):
        """
        Guarda cada DataFrame como archivo CSV.
        Permite revisar los datos en Excel antes de subirlos a la BD.
        ADICIÓN: esta funcionalidad no existe en el script de la escuela.
        """
        os.makedirs(folder, exist_ok=True)
        for name, df in self.dataframes.items():
            path = f'{folder}/{name}.csv'
            df.to_csv(path)  # index=True para incluir el ID como primera columna
            logging.info(f'  💾 {path} ({len(df):,} filas)')

    # ─────────────────────────────────────────────────────────
    # REPORTE FINAL
    # ─────────────────────────────────────────────────────────

    def generate_summary_report(self):
        """
        Genera un reporte final con estadísticas de la generación.
        Guarda el resultado en generation_summary.json.
        """
        # Reabrir conexión si fue cerrada por el bloque finally de la celda principal
        if LOAD_TO_DB:
            try:
                if self.connection is None or self.connection.closed:
                    self.connection = psycopg2.connect(**self.db_config)
                    logging.info('🔄 Reconexión para reporte final')
                if self.cursor is None or self.cursor.closed:
                    self.cursor = self.connection.cursor()
            except Exception as e:
                logging.warning(f'⚠️ No se pudo reconectar, validando sobre DataFrames: {e}')
                self.cursor = None
        logging.info('\n📊 RESUMEN DE GENERACIÓN DE DATOS')
        logging.info('=' * 50)

        total = sum(len(df) for df in self.dataframes.values())

        for name, df in self.dataframes.items():
            logging.info(f'  {name:15s}: {len(df):>10,} registros')
        logging.info(f'  {"TOTAL":15s}: {total:>10,} registros')

        # Estadística adicional: entregas por viaje
        if 'deliveries' in self.dataframes and 'trips' in self.dataframes:
            del_per_trip = (
                self.dataframes['deliveries']
                .groupby('trip_id').size()
            )
            logging.info(
                f'\n  Entregas/viaje: '
                f'AVG={del_per_trip.mean():.1f}, '
                f'MIN={del_per_trip.min()}, '
                f'MAX={del_per_trip.max()}'
            )

        all_valid = self.validate_data_quality()

        # Guardar resumen en JSON
        summary = {
            'generation_date':  datetime.now().isoformat(),
            'total_records':    total,
            'table_counts':     {k: len(v) for k, v in self.dataframes.items()},
            'validations_passed': all_valid
        }
        with open('generation_summary.json', 'w') as f:
            json.dump(summary, f, indent=2, default=str)

        logging.info('\n📄 Resumen guardado en generation_summary.json')
        return summary


print('✅ Clase DataGenerator definida correctamente')

✅ Clase DataGenerator definida correctamente


## Celda 4 — Ejecutar la generación de datos

Esta celda crea el objeto `DataGenerator` y ejecuta todas las generaciones en orden.

**Tiempo estimado:** ~3-5 minutos (la barra de progreso te muestra el avance)

## Celda 3b — Limpiar tablas (ejecutar solo si ya corriste el notebook antes)

Si ves errores de **llave duplicada**, corre esta celda primero para limpiar los datos anteriores. **No la ejecutes la primera vez.**

In [4]:
# ══════════════════════════════════════════════════════════════
# CELDA DE LIMPIEZA — Ejecutar ANTES de volver a correr el notebook
# Borra todos los datos previos para evitar errores de duplicados.
# ══════════════════════════════════════════════════════════════
import psycopg2

_conn = psycopg2.connect(**DB_CONFIG)
_cur  = _conn.cursor()

# TRUNCATE en orden inverso al de inserción (respeta foreign keys)
# RESTART IDENTITY reinicia los contadores de secuencia (IDs desde 1)
# CASCADE borra automáticamente registros dependientes
_cur.execute("""
    TRUNCATE TABLE
        maintenance,
        deliveries,
        trips,
        routes,
        drivers,
        vehicles
    RESTART IDENTITY CASCADE;
""")
_conn.commit()
_cur.close()
_conn.close()

print('🗑️  Tablas limpiadas correctamente — ya puedes ejecutar la Celda 4')


🗑️  Tablas limpiadas correctamente — ya puedes ejecutar la Celda 4


In [5]:
print('🚛 FLEETLOGIX - Generación de Datos Masivos')
print('=' * 60)
print('Objetivo: Generar 505,000+ registros respetando integridad referencial')
print('=' * 60)

# Crear el generador
generator = DataGenerator(DB_CONFIG)

try:
    # Conectar a PostgreSQL solo si está activo
    if LOAD_TO_DB:
        if not generator.connect():
            raise Exception('No se pudo conectar a PostgreSQL')

    # ── Generar datos en orden (respetando Foreign Keys) ──
    # Las tablas maestras primero, las transaccionales después
    print('\n[1/6] Generando vehículos...')
    generator.generate_vehicles(200)

    print('\n[2/6] Generando conductores...')
    generator.generate_drivers(400)

    print('\n[3/6] Generando rutas...')
    generator.generate_routes(50)

    print('\n[4/6] Generando viajes (100,000 registros, puede tardar ~2 min)...')
    generator.generate_trips(100_000)

    print('\n[5/6] Generando entregas (~400,000 registros, puede tardar ~3 min)...')
    generator.generate_deliveries(400_000)

    print('\n[6/6] Generando mantenimientos...')
    generator.generate_maintenance(5_000)

    print('\n✅ ¡Todos los datos generados exitosamente!')

except Exception as e:
    logging.error(f'❌ Error durante la generación: {e}')
    if generator.connection:
        generator.connection.rollback()  # Deshacer cambios parciales en la BD
    raise
finally:
    # finally se ejecuta SIEMPRE, aunque haya error
    # Garantiza que la conexión se cierre sin importar qué pasó
    if LOAD_TO_DB and generator.connection:
        generator.close()

2026-06-03 02:08:41,738 - INFO - ✅ Conexión exitosa a PostgreSQL
2026-06-03 02:08:41,741 - INFO - Generando 200 vehículos...
2026-06-03 02:08:41,830 - INFO - ✅ 200 vehículos generados


🚛 FLEETLOGIX - Generación de Datos Masivos
Objetivo: Generar 505,000+ registros respetando integridad referencial

[1/6] Generando vehículos...


2026-06-03 02:08:41,831 - INFO - Generando 400 conductores...



[2/6] Generando conductores...


2026-06-03 02:08:41,964 - INFO - ✅ 400 conductores generados
2026-06-03 02:08:41,966 - INFO - Generando 50 rutas...
2026-06-03 02:08:41,978 - INFO - ✅ 48 rutas generadas
2026-06-03 02:08:41,979 - INFO - Generando 100,000 viajes...


...


[5/6] Generando entregas (~400,000 registros, puede tardar ~3 min)...


Generando deliveries: 100%|█████████▉| 99949/100000 [00:52<00:00, 1919.97it/s]
2026-06-03 02:10:04,154 - INFO -   Progreso deliveries: 0/400,000
2026-06-03 02:10:14,895 - INFO -   Progreso deliveries: 50,000/400,000
2026-06-03 02:10:25,192 - INFO -   Progreso deliveries: 100,000/400,000
2026-06-03 02:10:34,734 - INFO -   Progreso deliveries: 150,000/400,000
2026-06-03 02:10:44,659 - INFO -   Progreso deliveries: 200,000/400,000
2026-06-03 02:10:54,736 - INFO -   Progreso deliveries: 250,000/400,000
2026-06-03 02:11:05,464 - INFO -   Progreso deliveries: 300,000/400,000
2026-06-03 02:11:16,398 - INFO -   Progreso deliveries: 350,000/400,000
2026-06-03 02:11:26,956 - INFO - ✅ 400,000 entregas generadas
2026-06-03 02:11:27,049 - INFO - Generando 5,000 registros de mantenimiento...



[6/6] Generando mantenimientos...


2026-06-03 02:11:27,938 - INFO - ✅ 4,908 mantenimientos generados
2026-06-03 02:11:27,941 - INFO - 🔒 Conexión cerrada



✅ ¡Todos los datos generados exitosamente!


## Celda 5 — Guardar CSVs y reporte final

In [6]:
# Guardar CSVs
if SAVE_CSV:
    print(f'💾 Guardando CSVs en {CSV_FOLDER}/')
    generator.save_csv(CSV_FOLDER)
    print('   Puedes abrir los .csv en Excel para revisar los datos')

# Reporte final
print()
summary = generator.generate_summary_report()
print(f'\n✅ JSON guardado en generation_summary.json')

2026-06-03 02:11:27,995 - INFO -   💾 ./data/vehicles.csv (200 filas)
2026-06-03 02:11:28,003 - INFO -   💾 ./data/drivers.csv (400 filas)
2026-06-03 02:11:28,006 - INFO -   💾 ./data/routes.csv (48 filas)


💾 Guardando CSVs en ./data/


2026-06-03 02:11:28,791 - INFO -   💾 ./data/trips.csv (100,000 filas)
2026-06-03 02:11:33,973 - INFO -   💾 ./data/deliveries.csv (400,000 filas)
2026-06-03 02:11:34,060 - INFO -   💾 ./data/maintenance.csv (4,908 filas)
2026-06-03 02:11:34,166 - INFO - 🔄 Reconexión para reporte final
2026-06-03 02:11:34,168 - INFO - 
📊 RESUMEN DE GENERACIÓN DE DATOS
2026-06-03 02:11:34,169 - INFO - ==================================================
2026-06-03 02:11:34,171 - INFO -   vehicles       :        200 registros
2026-06-03 02:11:34,173 - INFO -   drivers        :        400 registros
2026-06-03 02:11:34,175 - INFO -   routes         :         48 registros
2026-06-03 02:11:34,179 - INFO -   trips          :    100,000 registros
2026-06-03 02:11:34,181 - INFO -   deliveries     :    400,000 registros
2026-06-03 02:11:34,183 - INFO -   maintenance    :      4,908 registros
2026-06-03 02:11:34,184 - INFO -   TOTAL          :    505,556 registros
2026-06-03 02:11:34,240 - INFO - 
  Entregas/viaje: AV

   Puedes abrir los .csv en Excel para revisar los datos



2026-06-03 02:11:34,282 - INFO -   ✅ FK: Trips sin vehículo válido: 0
2026-06-03 02:11:34,660 - INFO -   ✅ FK: Deliveries sin trip válido: 0
2026-06-03 02:11:34,694 - INFO -   ✅ Temporal: arrival < departure: 0
2026-06-03 02:11:34,777 - INFO -   ✅ Peso: trips > capacidad vehículo: 0
2026-06-03 02:11:34,780 - INFO -   ✅ Deliveries sin tracking number: 0
2026-06-03 02:11:34,782 - INFO - 
📄 Resumen guardado en generation_summary.json



✅ JSON guardado en generation_summary.json


## Celda 6 — Exploración rápida de los datos generados

Muestra una muestra de cada tabla para verificar que los datos tienen sentido.

In [7]:
# Vista previa de cada tabla
for nombre, df in generator.dataframes.items():
    print(f'\n{'='*60}')
    print(f'📋 TABLA: {nombre.upper()}  ({len(df):,} registros)')
    print('='*60)
    display(df.head(3))  # display() en Jupyter muestra tablas formateadas


📋 TABLA: VEHICLES  (200 registros)


,license_plate,vehicle_type,capacity_kg,fuel_type,acquisition_date,status
vehicle_id,,,,,,
1,UDA-125,Van,1346.94,Gasolina,2023-10-04,active
2,HHE-204,Camión Grande,4173.88,Diésel,2022-04-16,active
3,VXR-130,Camión Mediano,2662.37,Diésel,2022-02-27,active



📋 TABLA: DRIVERS  (400 registros)


,employee_code,first_name,last_name,license_number,license_expiry,phone,hire_date,status
driver_id,,,,,,,,
1,EMP0001,Zoé,Núñez Osorio,7196439534,2027-04-07,5519314919,2021-10-07,active
2,EMP0002,Anabel,Tejada Porras,3667292043,2029-03-26,5551850671,2025-02-07,active
3,EMP0003,Ariadna,Rico Corral,6201313620,2028-04-09,5528498776,2023-09-13,active



📋 TABLA: ROUTES  (48 registros)


...

,vehicle_id,driver_id,route_id,departure_datetime,arrival_datetime,fuel_consumed_liters,total_weight_kg,status
trip_id,,,,,,,,
1,48,59,7,2024-06-03 10:33:00,2024-06-03 12:15:18.630175,12.70,2956.59,completed
2,96,147,26,2024-06-03 21:05:00,2024-06-04 08:30:26.287206,84.89,2435.57,completed
3,196,123,20,2024-06-03 16:43:00,2024-06-05 04:07:47.827505,195.61,4617.18,completed



📋 TABLA: DELIVERIES  (400,000 registros)


,trip_id,tracking_number,customer_name,delivery_address,package_weight_kg,scheduled_datetime,delivered_datetime,delivery_status,recipient_signature
delivery_id,,,,,,,,,
1,1,FL202600000001,César Luis Piña,"Calzada Puebla 904 Interior 259, Puebla",908.30,2024-06-03 10:45:47.328772,2024-06-03 12:39:47.328772,delivered,True
2,1,FL202600000002,Rosalia Carolina Quintana Armendáriz,"Corredor Comoras 110 Edif. 392 , Depto. 405, P...",524.62,2024-06-03 11:11:21.986316,2024-06-03 10:53:21.986316,delivered,True
3,1,FL202600000003,Iván Mauricio Rodríguez,"Calzada Jalisco 884 879, Puebla",824.65,2024-06-03 11:36:56.643859,2024-06-03 11:32:56.643859,delivered,True



📋 TABLA: MAINTENANCE  (4,908 registros)


,vehicle_id,maintenance_date,maintenance_type,description,cost,next_maintenance_date,performed_by
maintenance_id,,,,,,,
1,1,2024-07-02,Cambio de aceite,Cambio de aceite programado para 2024-07-02,5967.51,2024-08-01,José Eduardo Salinas
2,1,2024-07-26,Revisión de frenos,Revisión de frenos programado para 2024-07-26,9998.04,2024-09-24,Abigail Cuellar
3,1,2024-08-19,Cambio de aceite,Cambio de aceite programado para 2024-08-19,6124.88,2024-09-18,Aldonza Villareal


In [8]:
# Estadísticas de distribución
print('\n📊 DISTRIBUCIÓN DE VEHÍCULOS POR TIPO:')
display(generator.dataframes['vehicles']['vehicle_type'].value_counts().to_frame())

print('\n📊 DISTRIBUCIÓN DE ESTADOS DE ENTREGA:')
display(generator.dataframes['deliveries']['delivery_status'].value_counts().to_frame())

print('\n📊 DISTRIBUCIÓN DE ESTADOS DE VIAJES:')
display(generator.dataframes['trips']['status'].value_counts().to_frame())

print('\n📊 ENTREGAS POR VIAJE (estadísticas):')
del_por_viaje = generator.dataframes['deliveries'].groupby('trip_id').size()
display(del_por_viaje.describe().to_frame(name='entregas_por_viaje'))


📊 DISTRIBUCIÓN DE VEHÍCULOS POR TIPO:


,count
vehicle_type,
Camión Grande,63
Van,57
Camión Mediano,57
Motocicleta,23



📊 DISTRIBUCIÓN DE ESTADOS DE ENTREGA:


,count
delivery_status,
delivered,400000



📊 DISTRIBUCIÓN DE ESTADOS DE VIAJES:


,count
status,
completed,100000



📊 ENTREGAS POR VIAJE (estadísticas):


,entregas_por_viaje
count,99950.000000
mean,4.002001
std,1.096672
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,6.000000
